# 01.1 — CNV-W1A1: toate straturile în FPGA

Test pe **PYNQ**, cu pachetul BNN deja instalat și parametrii CIFAR-10.
Folosiți kernelul în care funcționează notebookul 01. Opriți kernelurile celorlalte notebookuri
QNN/BNN care folosesc FPGA: încărcarea CNV înlocuiește configurația curentă.

Cele 6 convoluții, 2 max-pooling și 3 straturi complet conectate sunt în FPGA.
CPU pregătește intrarea 32×32 RGB, gestionează transferurile și selectează clasa.
Nu reinstalăm BNN și nu reconstruim hardware-ul.

Măsurăm separat:
- **accelerator_s**: `usecPerImage / 1e6`, cronometrat în jurul `ExecAccel()`.
  Include pornirea și așteptarea pe CPU, exclude împachetarea și copiile host–accelerator.
  Nu este o măsurătoare exclusivă a ciclurilor FPGA.
- **api_s**: întregul apel `classify_cifar_details`, cu intrarea deja pregătită în RAM.
  Include citire CIFAR, împachetare, transferuri, accelerator, rezultate și mesajele bibliotecii.
- **pipeline_s**: de la citirea imaginii la clasa prezisă, inclusiv conversia CIFAR și
  crearea/ștergerea temporarului în RAM. Exclude încărcarea modelului și afișarea.

Două încălziri și cinci măsurători pe mod, câte o imagine per apel. Nu este benchmark video sau batch.
Temporarele folosesc numai `/dev/shm`; nu se salvează capturi sau jurnale.


In [ ]:
import gc, hashlib, os, platform, tempfile, time
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import bnn

print('BOARD:', os.environ.get('BOARD', 'not set'))
print('Platform:', platform.platform())
print('BNN:', bnn.__file__)
print('Parameters:', bnn.available_params(bnn.NETWORK_CNVW1A1))
if 'cifar10' not in bnn.available_params(bnn.NETWORK_CNVW1A1):
    raise RuntimeError('Missing CIFAR-10 CNV-W1A1 parameters.')
if not Path('/dev/shm').is_dir():
    raise RuntimeError('RAM-backed /dev/shm is required.')

IMAGE_NAME = 'deer.jpg'
EXPECTED_CLASS = 'deer'  # change with IMAGE_NAME; None for an unlabeled image
IMAGE_PATH_OVERRIDE = None
WARMUP_RUNS = 2
MEASURED_RUNS = 5
candidates = [Path.cwd() / 'pictures', Path.cwd() / 'notebooks/pictures',
              Path('/home/xilinx/jupyter_notebooks/bnn/pictures')]
candidates += [p / 'notebooks/pictures' for p in Path.cwd().parents]
image_path = (Path(IMAGE_PATH_OVERRIDE) if IMAGE_PATH_OVERRIDE is not None else
              next((p / IMAGE_NAME for p in candidates if (p / IMAGE_NAME).is_file()), None))
if image_path is None or not image_path.is_file():
    raise FileNotFoundError('Set IMAGE_PATH_OVERRIDE to an existing image, e.g. pictures/deer.jpg.')
image_path = image_path.resolve()
print('Input:', image_path)
print('SHA256:', hashlib.sha256(image_path.read_bytes()).hexdigest())
with Image.open(image_path) as opened:
    preview = opened.convert('RGB')
preview.thumbnail((480, 360))
display(preview)


## 1. Încărcarea modelului

Încărcarea bitstreamului și parametrilor este în afara benchmarkului.
Nu se creează un classifier software.


In [ ]:
if 'classifier' in globals():
    del classifier
    gc.collect()
t0 = time.perf_counter()
classifier = bnn.CnvClassifier(bnn.NETWORK_CNVW1A1, 'cifar10', bnn.RUNTIME_HW)
model_load_s = time.perf_counter() - t0
classes = list(classifier.classes)
if len(classes) != 10:
    raise ValueError('Expected 10 CIFAR-10 classes.')
print('Classes:', classes)
print('Model initialization: {:.3f} s'.format(model_load_s))
print('Runtime:', classifier.runtime)


## 2. Pregătirea intrării

Conversia reproduce `image_to_cifar` pentru imagini RGB: thumbnail 32×32 cu păstrarea
proporțiilor, centrare pe fundal alb, canale R/G/B consecutive după octetul de etichetă.
Imaginea originală nu este modificată. Scorurile CNV sunt întregi, nu probabilități.


In [ ]:
def prepare_record(path):
    with Image.open(path) as opened:
        image = opened.convert('RGB')
    resampling = Image.Resampling.LANCZOS if hasattr(Image, 'Resampling') else Image.LANCZOS
    image.thumbnail((32, 32), resampling)
    canvas = Image.new('RGB', (32, 32), (255, 255, 255))
    canvas.paste(image, ((32-image.width)//2, (32-image.height)//2))
    pixels = np.asarray(canvas, dtype=np.uint8)
    record = b'\x00' + pixels.transpose(2, 0, 1).tobytes()
    if len(record) != 3073:
        raise ValueError('Invalid CIFAR record size.')
    return record

def infer_prepared(path):
    t0 = time.perf_counter()
    scores = classifier.classify_cifar_details(str(path))
    api_s = time.perf_counter() - t0
    accelerator_s = float(classifier.usecPerImage) / 1e6
    scores = np.asarray(scores).copy()
    if scores.shape != (10,) or not np.all(np.isfinite(scores)):
        raise ValueError('Expected 10 finite CNV scores.')
    if not np.isfinite(accelerator_s) or accelerator_s <= 0 or accelerator_s > api_s:
        raise RuntimeError('Invalid native timing; check installed BNN timing units.')
    return dict(scores=scores, class_index=int(np.argmax(scores)),
                accelerator_s=accelerator_s, api_s=api_s)

def infer_from_image():
    t0 = time.perf_counter()
    record = prepare_record(image_path)
    prepared_at = time.perf_counter()
    with tempfile.TemporaryDirectory(prefix='cnv-bench-', dir='/dev/shm') as folder:
        path = Path(folder) / 'input.bin'
        path.write_bytes(record)
        result = infer_prepared(path)
    result['pipeline_s'] = time.perf_counter() - t0
    result['preprocess_s'] = prepared_at - t0
    return result

def check_same_scores(results, reference):
    for result in results:
        if not np.array_equal(result['scores'], reference):
            raise RuntimeError('Scores changed for identical input. Resolve this before interpreting timings.')


## 3. Intrare pregătită: accelerator și apel complet


In [ ]:
prepared_runs = []
with tempfile.TemporaryDirectory(prefix='cnv-bench-', dir='/dev/shm') as folder:
    path = Path(folder) / 'input.bin'
    path.write_bytes(prepare_record(image_path))
    for _ in range(WARMUP_RUNS):
        warmup = infer_prepared(path)
    for _ in range(MEASURED_RUNS):
        prepared_runs.append(infer_prepared(path))
reference_scores = prepared_runs[0]['scores'].copy()
check_same_scores(prepared_runs, reference_scores)
print('Prepared-input benchmark completed; RAM temporary removed.')


## 4. De la fișier imagine la clasa prezisă


In [ ]:
for _ in range(WARMUP_RUNS):
    warmup = infer_from_image()
pipeline_runs = [infer_from_image() for _ in range(MEASURED_RUNS)]
check_same_scores(pipeline_runs, reference_scores)
predicted = classes[int(np.argmax(reference_scores))]
print('Predicted:', predicted)
print('Expected:', EXPECTED_CLASS)
if EXPECTED_CLASS is not None:
    print('Correct:', predicted == EXPECTED_CLASS)
order = np.argsort(reference_scores)[::-1]
for i in order:
    print('{:12s} {:d}'.format(classes[i], int(reference_scores[i])))
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh([classes[i] for i in order], reference_scores[order])
ax.invert_yaxis()
ax.set_xlabel('CNV integer score (not probability)')
ax.set_title('CNV-W1A1 — CIFAR-10')
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. Timpi și contribuția acceleratorului

Intervalele sunt incluse unul în altul: **nu se adună**.
Diferența apel–accelerator include pregătirea nativă, transferuri, extragere și mesaje,
nu alte straturi de rețea pe CPU. Reciprocul timpului acceleratorului nu este FPS video.


In [ ]:
def summary(runs, key):
    values = np.asarray([row[key] for row in runs], dtype=float)
    return float(np.median(values)), float(values.min()), float(values.max())

measurements = [
    ('Accelerator call — ExecAccel', prepared_runs, 'accelerator_s'),
    ('Full API — prepared input', prepared_runs, 'api_s'),
    ('Image-to-class pipeline', pipeline_runs, 'pipeline_s'),
]
lines = ['| Measurement | Median (ms) | Min (ms) | Max (ms) |', '|---|---:|---:|---:|']
for name, runs, key in measurements:
    med, low, high = summary(runs, key)
    lines.append('| {} | {:.4f} | {:.4f} | {:.4f} |'.format(name, med*1000, low*1000, high*1000))
display(Markdown('\n'.join(lines)))

# Paired measurements from the same pipeline runs.
pipeline_median = summary(pipeline_runs, 'pipeline_s')[0]
accelerator_median = summary(pipeline_runs, 'accelerator_s')[0]
api_median = summary(pipeline_runs, 'api_s')[0]
fractions = [r['accelerator_s']/r['pipeline_s'] for r in pipeline_runs]
overheads = [r['pipeline_s']-r['accelerator_s'] for r in pipeline_runs]
print('Median accelerator share of pipeline: {:.2%}'.format(float(np.median(fractions))))
print('Median time outside accelerator call: {:.4f} ms'.format(float(np.median(overheads))*1000))
print('Equivalent sequential image-file rate: {:.2f} images/s (not camera FPS)'.format(1/pipeline_median))
print('Model loading excluded: {:.3f} s'.format(model_load_s))

fig, ax = plt.subplots(figsize=(9, 4))
names = [name for name, _, _ in measurements]
values = [summary(runs, key)[0]*1000 for _, runs, key in measurements]
ax.barh(names, values)
ax.invert_yaxis()
ax.set_xlabel('Median milliseconds — nested intervals, do not add')
for i, value in enumerate(values):
    ax.text(value, i, ' {:.4f} ms'.format(value), va='center')
ax.set_xlim(0, max(values)*1.35)
fig.tight_layout()
plt.show()
plt.close(fig)


## 6. Context: rezultatele anterioare

CNV clasifică **10 clase CIFAR-10, 32×32**. DoReFaNet/MobileNet clasifică **1.000 clase
ImageNet, 224×224**. Tabelul arată ordinul de mărime pentru aplicații diferite, nu un factor
de accelerare pentru aceeași sarcină/acuratețe. Pentru DoReFaNet PYNQ avem doar timpul
celor trei etape de calcul, nu un total end-to-end echivalent.


In [ ]:
lines = ['| Solution | Scope | Time (ms) |', '|---|---|---:|',
         '| CNV-W1A1 PYNQ — current | Accelerator call, all layers | {:.4f} |'.format(accelerator_median*1000),
         '| CNV-W1A1 PYNQ — current | Full API, same pipeline runs | {:.4f} |'.format(api_median*1000),
         '| CNV-W1A1 PYNQ — current | Image-file pipeline | {:.4f} |'.format(pipeline_median*1000),
         '| DoReFaNet PYNQ — previous | Three compute stages, CPU + FPGA | 1296.0 |',
         '| MobileNet RPi, 4 threads — previous | Entire network, OpenCV | 91.8 |',
         '| MobileNet RPi, 4 threads — previous | Image-file pipeline | 104.8 |']
display(Markdown('\n'.join(lines)))
print('Same scores in all measured CNV runs:', True)
print('No result files were saved. Save this notebook with outputs to share the measurements.')


## 7. Eliberarea resurselor

Fără resetarea dispozitivului sau încărcarea altui overlay. Pentru o nouă rundă, reexecutați inițializarea modelului.


In [ ]:
if 'classifier' in globals():
    del classifier
    gc.collect()
print('Classifier released. RAM temporary files were already removed.')


### Surse și validare
- [Straturile hardware: DoCompute](https://github.com/onigas/BNN-PYNQ/blob/master/bnn/src/network/cnvW1A1/hw/top.cpp)
- [Cronometrul ExecAccel: ramura hardware OFFLOAD, fără RAWHLS](https://github.com/onigas/BNN-PYNQ/blob/master/bnn/src/library/host/foldedmv-offload.h)
- [API și conversia CIFAR](https://github.com/onigas/BNN-PYNQ/blob/master/bnn/bnn.py)

Sintaxa și fluxul Python sunt verificate separat de hardware, cu un classifier simulat.
Inferența reală și timpii acestui notebook trebuie validați pe placa PYNQ.
